# Revisao: Ciclo de Dados com Pandas
Notebook simples para diagnosticar e limpar um dataset pequeno.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Base pequena com erros intencionais (contexto de entregas)
# Tempos entre 24 e 32 minutos representam entregas normais
# Tempo = 180 representa um valor extremo para demonstrar o tratamento de outlier
df_raw = pd.DataFrame([
    {"pedido_id": "P001", "tipo_entrega": "Moto",   "tempo_entrega": 24,  "valor_pedido": 45.0, "avaliacao": 5.0},
    {"pedido_id": "P002", "tipo_entrega": "Carro",  "tempo_entrega": 26,  "valor_pedido": np.nan, "avaliacao": 4.0},
    {"pedido_id": "P003", "tipo_entrega": "Moto",   "tempo_entrega": 28,  "valor_pedido": 62.0, "avaliacao": 5.0},
    {"pedido_id": "P004", "tipo_entrega": "Bike",   "tempo_entrega": 30,  "valor_pedido": 38.0, "avaliacao": 3.0},
    {"pedido_id": "P005", "tipo_entrega": "Carro",  "tempo_entrega": 32,  "valor_pedido": 80.0, "avaliacao": np.nan},
    {"pedido_id": "P006", "tipo_entrega": "Moto",   "tempo_entrega": 180, "valor_pedido": 55.0, "avaliacao": 2.0},
    {"pedido_id": "P004", "tipo_entrega": "Bike",   "tempo_entrega": 30,  "valor_pedido": 38.0, "avaliacao": 3.0}
])

df_raw

In [ ]:
print("Total de linhas:", len(df_raw))
print("Duplicadas:", df_raw.duplicated().sum())
print("Nulos por coluna:")
print(df_raw.isna().sum())

In [ ]:
# Limpeza didatica: remove duplicadas e preenche nulos numericos com mediana
df_clean = df_raw.drop_duplicates().copy()

for col in ["tempo_entrega", "valor_pedido", "avaliacao"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean

In [ ]:
# Exemplo simples de tratamento de outlier em tempo de entrega usando quantis (Q1 e Q3)

# Quantil e um ponto de corte da distribuicao:
# - quantile(0.25) => Q1: valor abaixo do qual estao ~25% dos dados
# - quantile(0.75) => Q3: valor abaixo do qual estao ~75% dos dados
q1 = df_clean["tempo_entrega"].quantile(0.25)
q3 = df_clean["tempo_entrega"].quantile(0.75)

# IQR (Intervalo Interquartil) mede a faixa central dos dados
# IQR = Q3 - Q1
iqr = q3 - q1

# Limites classicos para detectar outliers
# Valor abaixo de lower ou acima de upper e considerado discrepante
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# clip() limita os valores ao intervalo [lower, upper]
# Se o valor ja estiver dentro desse intervalo, ele nao muda
df_clean["tempo_entrega_tratado"] = df_clean["tempo_entrega"].clip(lower=lower, upper=upper)

# Verifica quantos valores foram realmente alterados
alterados = (df_clean["tempo_entrega"] != df_clean["tempo_entrega_tratado"]).sum()

print(f"Q1 (25%): {q1:.2f}")
print(f"Q3 (75%): {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Limite inferior: {lower:.2f}")
print(f"Limite superior: {upper:.2f}")
print(f"Tempos alterados pelo clipping: {alterados}")

if alterados == 0:
    print("Nenhum tempo foi alterado porque todos os valores ficaram dentro dos limites do IQR.")
    print("Com poucos registros, os limites podem ficar largos e nao capturar extremos.")

df_clean[["pedido_id", "tempo_entrega", "tempo_entrega_tratado"]]

In [ ]:
# Relacao entre colunas numericas
#
# Correlacao indica como duas variaveis mudam juntas.
# Valor proximo de +1: quando uma sobe, a outra tende a subir.
# Valor proximo de -1: quando uma sobe, a outra tende a descer.
# Valor proximo de 0: pouca relacao linear entre elas.
#
# O pandas usa por padrao a correlacao de Pearson:
# 1) compara variacao conjunta (covariancia)
# 2) normaliza pelo desvio padrao
# 3) retorna um valor sempre entre -1 e +1
#
# A diagonal da matriz sempre sera 1.0, porque cada variavel
# esta totalmente correlacionada com ela mesma.
df_clean[["tempo_entrega_tratado", "valor_pedido", "avaliacao"]].corr()